# Making drugs in silico - part IV

After setting up the umbrella sampling procedure, we will focus today on four points:
1. How to compute the free energy profile? The weighted histogram analysis method (WHAM)
2. How to check that the free energy profile is meaningful? Checking for convergence
3. How to estimate the error on the free energy? The bootstrapping method
4. How to compute the position-dependent diffusion coefficient using the set of biased simulations we have?

## 1. How to compute the free energy profile? The weighted histogram analysis method (WHAM)

**Note:** what if you want more theory? Here's a short list of ressources that describe in detail the relevant part of the theory in a clear way:
* <a href="http://doi.org/10.1021/ct100494z">doi.org/10.1021/ct100494z</a>
* <a href="http://doi.org/10.1002/wcms.66">doi.org/10.1002/wcms.66</a>
* <a href="http://doi.org/10.1016/S0010-4655(00)00215-0">doi.org/10.1016/S0010-4655(00)00215-0</a>

To compute the free energy profile from umbrella sampling procedure executed entirely with Gromacs, we can use the WHAM method that extracts the unbiased probability distribution of the system in order to compute the free energy profile along the given collective variable. For that you need to use the tool <code>gmx wham</code>


In [ ]:
import gromacs as gmx
import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import os, re
import pandas as pd

from scipy.optimize import curve_fit

In [ ]:
help(gmx.wham)

### 1.1 Let's start to look at the many options of <code>gmx wham</code>

1. <code>-bins</code> the number of bins, _i.e._ the number of points of free energy that will be estimated. A good starting guess is around 5 bin / window. You can plot the histogram of a window with different numbers to see that the minimal count is not too low.
2. <code>-temp</code> the temperature
3. <code>-tol</code> the tolerance used to assume that WHAM converged to a set of values of the free energy. A precision of 1e-5 is enough for a quick look at the profile. 1e-6 is the default and most common value. In some case, it makes sense to go up to 1e-8.
4. <code>-auto</code>, <code>-min</code>, <code>-max</code> the way to estimate the boundaries for which free energy values are computed. I would recommend to use the center of the first and last window by default.
5. <code>-b</code>, <code>-e</code> the beginning and ending time to be used. That is very important to produce a computation only on a converged part of the data and to check the convergence.
6. <code>-cycl</code> WHAM is applied with cyclic PBC, meaning that it assumes continuity between to 1st and last window. That is useful for system which have supposedly the same initial and final state (for instance, a molecule in the bulk water).
7. <code>-ix</code>, <code>-if</code> the input files to refer to the output files from Gromacs where the CV output or the forces output are stored (xvg files). These files should only have the path to the data files produced during the US procedure.
8. <code>-it</code> the path to the tpr. From the tpr, the pulling code is read by <code>gmx wham</code> to know the number of CVs and the values of the force constant for each window.
9. <code>-is</code> refers to a file mentioning which CVs should be used to compute the free energy.
10. <code>-o</code>, <code>-hist</code> the names of the output files.
11. <code>-bs-method</code>, <code>-nBootstrap</code>, the bootstrapping method is the one we will use to estimate the statistical error. We will use the default Bayesian implementation "b-hist". Generally, 100-200 bootstraps are enough. Bootstrapping method allows to estimate the statistical errors by making multiple (nbootstrap) guesses of the probability distribution. To obtain several guesses, random weights on the individual probability distributions of the different windows are applied. Mind that to not underestimate the error, one should have a very good overlap of multiple windows, **which is generally not the case**.


### 1.2. Tasks

1. Compute the free energy for the whole profile, for all the simulation long. Any comment?
2. Compute the free energy for the last 20 ns, but for each CV individually. Any comment?
3. Compute the free energy for the last 20 ns, for all the CVs at once, with the option </code>cycl</code>. How accurate do you think that result is?
4. Compute the free energy for the last 20 ns, in the same fashion as in 4., but estimate as well the statistical error using 100 bootstraps. Any comment?

### 1.3. About convergence

In many publications, you will find "convergence plots" for the free energy profiles, in the supporting information. These profiles give a qualitative estimate of the convergence of the free energy.

Let's plot one of this convergence plot. How would you use it to have a quantitative estimate of the convergence of the free energy values?

## 2. Estimating the position-dependent diffusion coefficient

As mentioned in the previous practical, we want to estimate the permeability coefficient of cisplatin in the model membrane of a sane cell. For that purpose, we need to estimate the position-dependent diffusion coefficient.

Here are two references which go through some of the methods used for that, how to implement them, and what is the underlying theory:
* <a href="http://10.1088/1367-2630/7/1/034">doi.org/10.1088/1367-2630/7/1/034</a> : 2005 article by G. Hummer which is one of the most referred work on the methods used for computing the position-dependent diffusion coefficient from biased MD simulations.
* <a href="https://doi.org/10.1021/ct300867e">doi.org/10.1021/ct300867e</a>: one possible extension to the work (re-)formulated by Hummer in 2005. 

We will focus on the formulation made by Hummer in 2005 of the method by Woolf and Roux published in 1994. The following section will use the formulation employed in this article. The diffusion coefficient $D(Z_i)$ for the window $i$ using a biasing potential centered in $Z=Z_i$ reads:

$$ D(Z_i) = \frac { var(Z) } { \tau_{Z} },$$

where $ var(Z) = \langle Z^2 \rangle - \langle Z \rangle^2 $ is the variance of the reaction coordinate Z for the given window $i$ and $ \tau_{Z} $ is the characteristic time of the autoccorelation function defined as:

$$ \tau_{Z} = \frac {1} {var(Z)} \int_{t=0}^{t=\infty} \langle \delta Z(t)\delta Z(0) \rangle dt $$,

with $ \delta Z(t) = Z(t) - \langle Z \rangle $.

The last task of this practical is to implement the computation of the diffusion coefficient as written in the following article: <a href="http://doi.org/10.3389/fphar.2018.00026">doi.org/10.3389/fphar.2018.00026</a>. As it is not a trivial task, you may choose to work altogether if you find it easier.

In [ ]:
%pwd

In [ ]:
%cd ../02.permeability/
%ls

In [ ]:
# Define the path to the input files
folder_path = '01.results'
# Define the pattern of the names of the input files
file_pattern = re.compile(r'win_(\d{2})\.xvg')
# Get a list of all those files with xvg extension
files = 
# Extract and sort the integers from the file names
sequence_list = 
# Rewrite the list of files sequentially
files = 
# Define column names
columns = ["time", "CIS1", "CIS2", "CIS3"]
# Specify the maximum number of points for the autocorrelation function
acf_size = 
# List of "s-values" to use in the reciprocal space after Laplace transform
# These values are simply inverse time
s_vals = 
# Define the fitting function
def fit_fn(XXXXX):
    return XXXXX
## Loop through files
# Create an empty list to store DataFrames for each file
dfs = []
# Defining the list of diffusion coefficient values and the one of the CV values that will be appended through the following loop
D = []
cv = []
for win,file in enumerate(files):
#for win,file in enumerate(['win_01.xvg']):
    ##### Reading the XVG file of the current window #####
    # Construct the file path
    file_path = os.path.join(folder_path, file)
    # Read the xvg file using Pandas, skipping the header lines
    with open(file_path, 'r') as xvg_file:
        header_lines = 0
        for line in xvg_file:
            if line.startswith('@') or line.startswith('#'):
                header_lines += 1
            else:
                break             
    df = pd.read_csv(file_path, delim_whitespace=True, skiprows=header_lines, comment='&', usecols=[0, 1, 2, 3], names=columns)
    # Append the DataFrame to the list
    dfs.append(df)
    ######################################################
    ############ Computing the value of the CV ###########
    cv.append(XXXXX)
    print(f'Analyzing window {sequence_list[win]}.\n\tCV = {cv[-1]:.1f} nm')
    ######################################################
    ######## Cutting the dataset to a maximum time #######
    limit = 2.0e4 # 20 ns
    condition = dfs[win]['time'] < limit
    dfs[win] = dfs[win][condition]
    ######################################################
    ######## Find the Delta t between two successive data #######
    dt = XXXXX-XXXXX
    print(f'\tdt = {dt} ps')
    ######################################################
    ######## Compute the individual diffusion coefficient for each cisplatin molecule #######
    D_val = []
    for cis_index,cis_name in enumerate(columns[1:]):
        print(f'\tComputing D(z) for {cis_name}.')
        # Compute average and variance
        cv_average = np.mean(XXXXX)
        cv_variance = np.var(XXXXX)
        print(f'\tmean = {cv_average:.5f}\n\tvar = {cv_variance:.5f}')
        # Compute the time autocorrelation function of dfs[win][cis_name]
        np.savetxt('tmp.xvg', np.c_[dfs[win]['time'],dfs[win][cis_name]],fmt='%1.6e')
        gmx.analyze(f = 'tmp.xvg', ac = 'autocorr.xvg', acflen = XXXXX, nonormalize = True, subav = True)
        # Load the file and delete it
        acf = np.loadtxt('autocorr.xvg', comments=["#","@","&"])
        os.remove('autocorr.xvg')
        # Plot the ACF to check it: comment if need be
        plt.figure(figsize=(10, 6))
        plt.plot(acf[:, 0], acf[:, 1], label=f'{cis_name}')
        plt.title(f'Autocorrelation Function ({cis_name}, Window {win})')
        plt.xlabel('Time Lag')
        plt.ylabel('Autocorrelation')
        plt.legend()
        plt.show()
        # Compute integrals to do the Laplace transform for a set of s-values
        print(f'\tCompute integral')
        # Initialize the list
        int_vals = np.zeros(len(s_vals))
        # Integrate
        XXXXX
        # Fit to s_vals a/(s+b) to extrapolate tau_i(s=0)
        popt,pcov = curve_fit(XXXXX)
        # Print tau
        tau = XXXXX
        print(f'\tThe autocorrelation characteristic time is {tau} ps')
        # Compute the local diffusion coefficient
        Dlocal = XXXXX
        # Append D_val
        D_val.append(Dlocal)
    # Compute the average diffusion coefficient and append to the list
    D.append(np.mean(D_val))
    print(f'\tThe average local diffusion coefficient D = {D[-1]} +/- {np.std(D_val)} m^2 s^-1.')
# Plot the position-dependent diffusion coefficient
plt.figure(figsize=(10, 6))
plt.plot(cv, D)
plt.title('Position-dependent diffusion coefficient')
plt.xlabel('CV, nm')
plt.ylabel('Diffusion coefficient, m^2 s^-1')
plt.legend()
plt.show()
# Clean the folder
os.remove('tmp.xvg')

In [ ]:
# Saving it to file
data = np.column_stack((cv, D))
output_file_path = 'diffusion.xvg'
np.savetxt(output_file_path, data, delimiter='\t')

Now that we have free energy and diffusion profiles, we simply need to compute the permeability coefficient by integrating $D(Z)/e^{\beta \Delta G(Z)}$ for $Z \in \left[-3.5, 3.5\right]$.

In [ ]:
# Load the spline interpolation of each plots
diffusion_spline = 'diffusion_spline.dat'
free_energy_spline = 'free_energy_spline.dat'
diff_data = np.loadtxt(diffusion_spline, comments=["#","@","&"])
free_data = np.loadtxt(free_energy_spline, comments=["#","@","&"])
#print(diff_data[:,0])
# Compute the constant to multiply by the FE in the exp
constant = XXXXX
# Compute the permeability list
resistance_local = XXXXX
# Plotting the local resistance v. CV
plt.figure(figsize=(10, 6))
plt.plot(free_data[:,0], resistance_local)
plt.xlabel('CV, nm')
plt.ylabel('Resistance, s^1 m^-2')
plt.yscale('log')
plt.legend()
plt.show()
# Compute the dt
step_int = XXXXX
# Integrate local resistance
resistance = np.trapz(XXXXX)
permeability = 1 / resistance
print(f'The permeability coefficient P={permeability} m s^-1.')